# NEXUS Wake Word + Command Intent Training (Hybrid Approach)

## Strategy: 4 Category Models + STT Fallback

This notebook trains **4 category-level acoustic models** that detect the *type* of command, then uses STT to extract the *parameter* (which app, what search query). This gives fast response for known command patterns while remaining flexible for any app.

### Architecture

```
User says "nexus open gmail"
  → Wake word detects "nexus" (acoustic, ~0ms)
  → Category model detects "open" intent (acoustic, ~80ms)
  → Record 1-2s audio (shorter — just need the parameter)
  → STT returns "gmail"
  → Intent parser → execute

User says "nexus do something weird"
  → Wake word detects "nexus"
  → No category model fires
  → Record 3s audio → STT → intent parser → execute or send to backend
```

### Models Trained

| Model | Detects | Example phrases | Output |
|-------|---------|-----------------|--------|
| `command_open.onnx` | "open/launch/start **" | "open gmail", "launch spotify", "start vs code" | `open_app` intent |
| `command_close.onnx` | "close/quit/exit **" | "close discord", "quit chrome", "exit terminal" | `close_app` intent |
| `command_search.onnx` | "search/find/look up **" | "search for cats", "find docs", "look up python" | `search` intent |
| `command_play.onnx` | "play/listen to **" | "play music", "listen to jazz", "play on spotify" | `play` intent |

### Resource Usage

| Metric | Value |
|--------|-------|
| Training time | ~1.5 hours on Colab (4 × ~20 min) |
| Model files | 4 × 800KB = 3.2 MB |
| RAM (models) | ~8 MB |
| RAM (total NEXUS) | ~58 MB (well under 250 MB target) |
| CPU | 4 inferences per 80ms chunk (negligible) |
| Latency: wake → execute | 300-800ms (shorter STT — only parameter) |
| Works for new apps | Yes (STT extracts any app name) |

### Training Data Per Model

Each category model is trained with:
- **Positive samples**: 500+ TTS clips of the command pattern with different apps
  - e.g. for `command_open`: "open gmail", "open spotify", "open chrome", "launch figma", "start discord", etc.
  - Uses all 75+ apps from the laptop scan × multiple verb variants
- **Negative samples**: 2000+ clips of other command types, random phrases, silence, noise
  - e.g. for `command_open` negatives: "close gmail", "search for gmail", "play gmail"
- **Hard negatives**: Soundalike phrases that should NOT trigger
  - e.g. "open" → "oh pen", "oh pin", "ope in", "oppen"

### Apps Covered (from laptop scan — 75+ apps)

Gmail, Google Chrome, Google Gemini, Google Meet, Google Drive, Google Photos,
Google Colab, YouTube, Spotify, Netflix, WhatsApp, Discord, Signal, Telegram,
LinkedIn, Microsoft Edge, Teams, Word, Excel, PowerPoint, OneNote, OneDrive,
Outlook, Paint, Visual Studio Code, Android Studio, Blender, DaVinci Resolve,
Figma, Canva, Steam, Rockstar Games, MongoDB Compass, Postman, Cursor, Brave,
Claude, ChatGPT, Perplexity, Copilot, Devin, Trae, Zed, Kiro, Jules, Lovable,
OpenAI Codex, Notion, Obsidian, Slack, Zoom, GitHub, GitLab, Firebase, Vercel,
Netlify, Spline, LiveCaptions, WPS Office, Sheets, iCloud, Calculator, Calendar,
Clock, Weather, Settings, Camera, Photos, Notepad, Terminal, PowerShell,
File Explorer, Task Manager, Snipping Tool, Maps, News, Bing, Xbox, Solitaire

## Instructions

1. **Runtime → Change runtime type → T4 GPU** (or L4 GPU + High RAM on Colab Pro)
2. **Runtime → Run all**
3. Wait ~1.5 hours for all 4 models to train
4. Download the 4 `.onnx` files + `command_intents.json`
5. Place them in `src-tauri/resources/oww/commands/`:
   - `commands/command_open.onnx`
   - `commands/command_close.onnx`
   - `commands/command_search.onnx`
   - `commands/command_play.onnx`
   - `commands/command_intents.json`
6. Rebuild NEXUS with `cargo build --release --features custom-protocol`

## Wake Word Status

The existing `nexus.onnx` wake word model **already works** after the 32768x
scaling fix in `wakeword_oww.rs`. It detected "NEXUS" with probability 0.991.
No wake word retraining is needed unless you want to add soundalike tolerance.

In [ ]:
# =============================================================================
# APP LIST — All user-facing apps found on the laptop (75+ apps)
# These are used to generate training phrases for the 4 category models.
# =============================================================================

APPS = [
    # Google ecosystem
    "gmail", "google chrome", "chrome", "google gemini", "gemini",
    "google meet", "meet", "google drive", "drive", "google photos",
    "google colab", "colab", "google cloud console", "cloud console",
    # Media & entertainment
    "youtube", "spotify", "netflix", "whatsapp",
    # Communication
    "discord", "signal", "telegram", "linkedin", "teams",
    "zoom", "slack", "skype",
    # Microsoft Office
    "microsoft edge", "edge", "microsoft word", "word",
    "microsoft excel", "excel", "microsoft powerpoint", "powerpoint",
    "microsoft onenote", "onenote", "microsoft onedrive", "onedrive",
    "microsoft outlook", "outlook", "microsoft paint", "paint",
    "microsoft to do", "to do", "microsoft power bi", "power bi",
    # Development tools
    "visual studio code", "vs code", "code", "android studio",
    "blender", "davinci resolve", "figma", "canva",
    "mongodb compass", "compass", "postman", "cursor",
    # AI tools
    "claude", "chatgpt", "gpt", "perplexity", "copilot",
    "devin", "trae", "zed", "kiro", "jules", "lovable",
    "openai codex", "codex",
    # Gaming
    "steam", "rockstar games", "xbox", "solitaire",
    # Browsers
    "brave", "firefox", "opera",
    # Dev platforms
    "github", "gitlab", "firebase", "vercel", "netlify",
    "stack overflow",
    # Other
    "spline", "livecaptions", "wps office", "sheets",
    "icloud", "notion", "obsidian", "trello", "jira",
    # Windows built-in
    "calculator", "calendar", "clock", "weather",
    "settings", "camera", "photos", "notepad", "wordpad",
    "terminal", "command prompt", "powershell",
    "file explorer", "explorer", "task manager",
    "snipping tool", "screenshot", "maps", "news",
    "bing", "store", "widgets", "command palette",
    "microsoft store", "app store",
    "google", "search",
]

# Deduplicate
APPS = sorted(set(APPS))
print(f"Total unique app names: {len(APPS)}")

# =============================================================================
# CATEGORY MODELS — 4 models, each detects a command TYPE
# Each model is trained with ALL apps as positive samples, and ALL other
# command types as negative samples. This way the model learns the PATTERN
# "open <anything>" not specific app names.
# =============================================================================

# Verbs for each command category
COMMAND_VERBS = {
    "open":   ["open", "launch", "start", "bring up", "show me", "pull up", "fire up", "load"],
    "close":  ["close", "quit", "exit", "kill", "shut down", "stop", "terminate", "end"],
    "search": ["search for", "look up", "find", "google", "query", "search"],
    "play":   ["play", "listen to", "start playing", "put on", "stream"],
}

# Soundalike negatives for each verb pattern
VERB_SOUNDALIKES = {
    "open":   ["oh pen", "oh pin", "ope in", "oppen", "oh pan", "upen", "openn"],
    "close":  ["cloze", "clo s", "klose", "clows", "cloze", "claws", "closee"],
    "search": ["serch", "surch", "searchh", "circh", "sirch", "serchh", "search for"],
    "play":   ["pley", "pleigh", "plai", "plae", "pley", "pray", "playy"],
}

# Generate positive phrases for each category (all apps × all verbs)
def generate_category_phrases():
    categories = {}
    for cat, verbs in COMMAND_VERBS.items():
        phrases = []
        for verb in verbs:
            for app in APPS:
                phrases.append(f"{verb} {app}")
        categories[cat] = phrases
    return categories

CATEGORY_PHRASES = generate_category_phrases()

# Generate negative phrases (other command types + soundalikes)
def generate_negatives(target_cat):
    negatives = []
    # Other command types as negatives
    for cat, phrases in CATEGORY_PHRASES.items():
        if cat != target_cat:
            negatives.extend(phrases[:50])  # 50 samples from each other category
    # Soundalike negatives
    negatives.extend(VERB_SOUNDALIKES[target_cat])
    # Common non-command phrases
    negatives.extend([
        "hey nexus", "ok nexus", "nexus", "nexus wake up",
        "hey siri", "ok google", "alexa", "hey cortana",
        "what time is it", "how is the weather", "tell me a joke",
        "thank you", "good morning", "good night", "hello",
        "yes", "no", "maybe", "please", "sorry",
    ])
    return negatives

print(f"\n=== Category phrases ===")
for cat, phrases in CATEGORY_PHRASES.items():
    negs = generate_negatives(cat)
    print(f"  {cat}: {len(phrases)} positives, {len(negs)} negatives")
    print(f"    Sample positives: {phrases[:3]}")
    print(f"    Sample negatives: {negs[:3]}")

total_pos = sum(len(p) for p in CATEGORY_PHRASES.values())
print(f"\nTotal positive phrases across all categories: {total_pos}")
print(f"Total models to train: 4")

## 1. Comprehensive install — apt + pip + verify

**Python 3.13 compatible** (Colab Aug 2026). Key changes from old notebook:
- `piper_phonemize` from k2-fsa mirror (has cp313 wheels) — the old `piper-phonemize-cross` doesn't exist for Python 3.13
- `piper-tts==1.3.0 --no-deps` (uses onnxruntime, not piper-phonemize)
- Added `torch.load` weights_only=False patch for PyTorch 2.6+ (piper-sample-generator loads .pt checkpoints)

In [ ]:
# Native deps
!apt-get update -qq 2>&1 | tail -1
!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

# ─── Python deps — Python 3.13 compatible install order ─────────────
# (a) piper_phonemize from k2-fsa mirror FIRST (has cp313 wheels)
#     The old piper-phonemize-cross doesn't exist for Python 3.13.
#     The k2-fsa mirror has v1.4.7 with cp313 manylinux wheels.
!pip install -q piper_phonemize -f https://k2-fsa.github.io/icefall/piper_phonemize.html

# (b) All openwakeword.train / data.py / utils.py transitive deps
!pip install -q \
    webrtcvad \
    mutagen==1.47.0 \
    torchinfo \
    torchmetrics \
    pyyaml \
    tqdm \
    datasets \
    soundfile \
    audiomentations \
    torch_audiomentations \
    pronouncing \
    onnxruntime \
    onnx \
    speechbrain \
    acoustics \
    scipy \
    requests \
    huggingface_hub

# (b2) deep-phonemizer — needed by openwakeword data.py for adversarial text generation
#     sdist only (no wheels) but pure Python, builds fine on 3.13
#     Install with --no-deps to avoid tensorboard conflicts
!pip install -q --no-deps deep-phonemizer==0.0.19

# (c) piper-tts 1.3.0 LAST via --no-deps (uses onnxruntime, not piper-phonemize)
#     This avoids dependency conflicts with piper_phonemize versions.
!pip install -q --no-deps piper-tts==1.3.0

# ─── Verify imports — every module openwakeword's train.py touches ────
import sys
print('Python:', sys.version)
import torch, torchinfo, torchmetrics, scipy, numpy
print(f'  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
from tqdm import tqdm
import yaml, mutagen, pronouncing
import torchaudio, audiomentations, torch_audiomentations
import speechbrain, acoustics
import onnx, onnxruntime, soundfile, requests
from piper_phonemize import phonemize_espeak
from piper import PiperVoice, SynthesisConfig
try:
    from dp.phonemizer import Phonemizer
    print('  deep-phonemizer: OK')
except ImportError:
    print('  WARNING: deep-phonemizer not available — adversarial text generation will be skipped')
print('  All deps import cleanly.')

# ─── PyTorch 2.6+ fix: torch.load weights_only default changed ───────
# piper-sample-generator uses torch.load() to load the .pt model.
# PyTorch 2.6+ changed the default from weights_only=False to True,
# which breaks loading the Piper PyTorch checkpoint.
# Fix: monkey-patch torch.load to default weights_only=False.
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
print('  torch.load patched: weights_only defaults to False (PyTorch 2.6+ fix)')


## 1b. Keep-alive heartbeat (prevents Colab idle timeout)

Colab free tier disconnects after ~90 minutes of no interaction. The `heartbeat()` function below prints a status line at most once per minute, which counts as "interaction" in the Colab UI and prevents idle disconnects during long training runs.

**No Google Drive required.** All models are saved to `/content/` (Colab's local storage) and downloaded via the browser when training completes.

In [ ]:
import time

# No Google Drive mounting — this notebook works entirely in /content/.
# Models are saved locally and downloaded via the browser.

# Keep-alive helper: call this during long training to prevent idle timeout.
# Colab disconnects after ~90 min of no interaction. This prints a heartbeat
# which counts as "interaction" in the Colab UI.
_last_heartbeat = time.time()
def heartbeat(label=''):
    global _last_heartbeat
    now = time.time()
    elapsed = now - _last_heartbeat
    if elapsed > 60:  # Print at most once per minute
        print(f'  [heartbeat] {time.strftime("%H:%M:%S")} — {label} (alive, {elapsed:.0f}s since last)', flush=True)
        _last_heartbeat = now

print('OK heartbeat ready — no Google Drive needed')

In [ ]:
import os, sys
os.chdir('/content')

# piper-sample-generator (pinned to flat-layout commit)
PSG_DIR = '/content/piper-sample-generator'
PSG_PIN = '1a8c49bd29b3a132721086ee88f2253f788594a8^'
PSG_GS  = f'{PSG_DIR}/generate_samples.py'
if not os.path.exists(PSG_GS):
    !rm -rf {PSG_DIR}
    !git clone -q https://github.com/rhasspy/piper-sample-generator {PSG_DIR}
!cd {PSG_DIR} && git fetch -q --all && git checkout -q {PSG_PIN}
assert os.path.exists(PSG_GS), 'piper-sample-generator pin failed'
print(f'  OK piper-sample-generator')

# libritts model (~200 MB)
PIPER_MODEL = f'{PSG_DIR}/models/en_US-libritts_r-medium.pt'
PIPER_MODEL_URL = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    !mkdir -p {PSG_DIR}/models
    !wget -q --tries=5 --timeout=300 -O {PIPER_MODEL} {PIPER_MODEL_URL}
assert os.path.getsize(PIPER_MODEL) > 100_000_000
print(f'  OK libritts model: {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB')

# openwakeword
# Use --no-deps because speexdsp_ns is not available on Colab.
# All dependencies are installed separately in cell 3.
OWW_DIR = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'
if not os.path.exists(OWW_TRAIN):
    !rm -rf {OWW_DIR}
    !git clone -q https://github.com/dscripka/openwakeword {OWW_DIR}
    !pip install -q --no-deps -e {OWW_DIR}
assert os.path.exists(OWW_TRAIN)
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]
import openwakeword
assert openwakeword.__file__ is not None
print(f'  OK openwakeword: {openwakeword.__file__}')

## 3. Apply runtime patches (8 patches)

1. `torchaudio.set_audio_backend` → no-op (removed in torchaudio 2.x)
2. Copy `generate_samples.py` from piper-sample-generator into openwakeword
3. HuggingFace Hub timeouts → 120s (for slow Colab network)
4. `torchaudio.info()` shim (uses soundfile instead)
5. `generate_samples` model arg default (so it doesn't crash if not passed)
6. `train.py` val dtype cast (`.float()` to avoid dtype mismatch)
7. `data.py` graceful fallback for deep-phonemizer failures (catches ALL exceptions)
8. `dp/model/model.py` torch.load `weights_only=False` (PyTorch 2.6+ fix for DeepPhonemizer checkpoint)

In [ ]:
import os, glob

# Patch A: torchaudio.set_audio_backend → pass
for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    !sed -i 's|torchaudio.set_audio_backend("soundfile")|pass  # patched|' "{path}"

# Patch B: copy generate_samples.py
src = '/content/piper-sample-generator/generate_samples.py'
dst = '/content/openwakeword/openwakeword/generate_samples.py'
if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
    !cp "{src}" "{dst}"

# Patch C: HF Hub timeouts
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
import huggingface_hub.constants as hfc
for attr in ['DEFAULT_ETAG_TIMEOUT', 'DEFAULT_DOWNLOAD_TIMEOUT',
             'HF_HUB_ETAG_TIMEOUT', 'HF_HUB_DOWNLOAD_TIMEOUT']:
    if hasattr(hfc, attr): setattr(hfc, attr, 120)

# Patch D: torchaudio.info shim
import torchaudio
init_path = torchaudio.__file__
SHIM_MARKER = '# PATCH: info() shim for torchaudio 2.x'
with open(init_path) as f:
    content = f.read()
if SHIM_MARKER not in content:
    shim = (f'\n\n{SHIM_MARKER}\n'
            'def info(file_path, *args, **kwargs):\n'
            '    import soundfile as _sf\n'
            '    si = _sf.info(str(file_path))\n'
            '    return type("_TorchaudioInfo", (), {\n'
            '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
            '        "num_channels": si.channels, "bits_per_sample": 16,\n'
            '        "encoding": "PCM_S",\n'
            '    })()\n')
    with open(init_path, 'a') as f: f.write(shim)
    import importlib; importlib.reload(torchaudio)

# Patch E: generate_samples model arg default
TARGET = '/content/piper-sample-generator/generate_samples.py'
!sed -i 's|model: Union\[str, Path\],|model: Union[str, Path] = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",|' "{TARGET}"
!cp "{TARGET}" /content/openwakeword/openwakeword/generate_samples.py

# Patch F: train.py val dtype cast
TRAIN_PY = '/content/openwakeword/openwakeword/train.py'
!sed -i 's|val_predictions = self.model(x_val)$|val_predictions = self.model(x_val.float())|' "{TRAIN_PY}"

# Patch G: data.py — graceful fallback if deep-phonemizer (dp) fails for ANY reason
#
# data.py's generate_adversarial_texts() has `from dp.phonemizer import Phonemizer`
# INSIDE the function body. The import may succeed, but Phonemizer.from_checkpoint()
# may fail (e.g. torch.load weights_only error on PyTorch 2.6+).
#
# Strategy: rename the original function and add a wrapper that catches ALL
# exceptions. If anything fails, return [] (skip adversarial generation).
# Our custom_negative_phrases already provide good negatives.
DATA_PY = '/content/openwakeword/openwakeword/data.py'
if os.path.exists(DATA_PY):
    with open(DATA_PY) as f:
        data_content = f.read()

    # Check if already patched (look for our wrapper marker)
    if '_generate_adversarial_texts_orig' not in data_content:
        # Step 1: Rename the original function so we can wrap it
        data_content = data_content.replace(
            'def generate_adversarial_texts(',
            'def _generate_adversarial_texts_orig('
        )
        # Step 2: Append a wrapper function at the end of the file
        # The wrapper catches ALL exceptions (ImportError, UnpicklingError, etc.)
        data_content += '''

# Patched wrapper: skip adversarial generation if deep-phonemizer fails for any reason
def generate_adversarial_texts(*args, **kwargs):
    try:
        return _generate_adversarial_texts_orig(*args, **kwargs)
    except Exception as e:
        print(f"WARNING: adversarial text generation failed ({e}), skipping")
        return []
'''
        with open(DATA_PY, 'w') as f:
            f.write(data_content)
        print('  Patched data.py: wrapper function for deep-phonemizer fallback')
    else:
        print('  data.py already patched (wrapper found)')

    # Verify the patch worked: data.py must be valid Python
    import py_compile
    try:
        py_compile.compile(DATA_PY, doraise=True)
        print('  data.py: syntax OK')
    except py_compile.PyCompileError as e:
        print(f'  ERROR: data.py has syntax error after patching: {e}')
        raise

# Patch H: dp/model/model.py — fix torch.load weights_only for PyTorch 2.6+
# DeepPhonemizer's load_checkpoint() calls torch.load() without weights_only=False.
# PyTorch 2.6+ changed the default to True, which breaks loading the .pt checkpoint.
# The checkpoint is from a trusted public S3 bucket (DeepPhonemizer models).
DP_MODEL_PY = '/usr/local/lib/python3.13/dist-packages/dp/model/model.py'
if os.path.exists(DP_MODEL_PY):
    with open(DP_MODEL_PY) as f:
        dp_content = f.read()
    if 'weights_only=False' not in dp_content:
        # Add weights_only=False to the torch.load call
        dp_content = dp_content.replace(
            'checkpoint = torch.load(checkpoint_path, map_location=device)',
            'checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)'
        )
        with open(DP_MODEL_PY, 'w') as f:
            f.write(dp_content)
        print('  Patched dp/model/model.py: torch.load weights_only=False')
    else:
        print('  dp/model/model.py already patched')
else:
    print('  WARNING: dp/model/model.py not found — deep-phonemizer may not be installed')

print('All 8 patches applied.')

## 4. Download shared data (MIT RIRs, FMA, ACAV100M — done ONCE)

In [ ]:
import os, time, shutil

# Shared OWW models (melspectrogram + embedding)
OWW_MODELS_DIR = '/content/oww_models'
os.makedirs(OWW_MODELS_DIR, exist_ok=True)
for fname in ['melspectrogram.onnx', 'embedding_model.onnx']:
    path = f'{OWW_MODELS_DIR}/{fname}'
    if not os.path.exists(path) or os.path.getsize(path) < 100_000:
        url = f'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{fname}'
        !wget --tries=5 --timeout=120 -O {path} {url}
    sz = os.path.getsize(path)
    assert sz > 100_000, f'{fname} download failed: only {sz} bytes'
    print(f'  OK {fname}: {sz/1e6:.1f} MB')

# ─── CRITICAL: Copy OWW models to openWakeWord's resources directory ───────
# train.py's compute_features_from_generator() calls AudioFeatures() which
# looks for models at:
#   /content/openwakeword/openwakeword/resources/models/melspectrogram.onnx
# But we downloaded them to /content/oww_models/. Without this copy,
# augmentation fails with NoSuchFile error.
OWW_RESOURCES = '/content/openwakeword/openwakeword/resources/models'
os.makedirs(OWW_RESOURCES, exist_ok=True)
for fname in ['melspectrogram.onnx', 'embedding_model.onnx']:
    src = f'{OWW_MODELS_DIR}/{fname}'
    dst = f'{OWW_RESOURCES}/{fname}'
    if not os.path.exists(dst) or os.path.getsize(dst) < 100_000:
        shutil.copy2(src, dst)
        print(f'  Copied {fname} → {dst}')
    else:
        print(f'  Already exists: {dst}')
# Verify
for fname in ['melspectrogram.onnx', 'embedding_model.onnx']:
    assert os.path.exists(f'{OWW_RESOURCES}/{fname}'), f'Missing {fname} in OWW resources!'
print(f'  OK OWW models in resources directory')

# MIT RIRs
RIR_DIR = '/content/mit_rirs'
if not os.path.exists(RIR_DIR) or len(os.listdir(RIR_DIR)) < 250:
    from huggingface_hub import snapshot_download
    for attempt in range(6):
        try:
            snapshot_download('davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', local_dir='/content/mit_rirs_raw')
            break
        except Exception as e:
            print(f'  MIT RIRs retry {attempt+1}: {e}')
            time.sleep(5)
    os.makedirs(RIR_DIR, exist_ok=True)
    import soundfile as sf, glob
    from scipy.signal import resample_poly
    for f in glob.glob('/content/mit_rirs_raw/**/*.wav', recursive=True):
        data, sr = sf.read(f)
        if sr != 16000:
            data = resample_poly(data.astype('float32'), 16000, sr)
        sf.write(f'{RIR_DIR}/{os.path.basename(f)}', data, 16000)
    # Clean up raw download to save ~0.5 GB disk
    shutil.rmtree('/content/mit_rirs_raw', ignore_errors=True)
    print('  Cleaned up mit_rirs_raw (saved ~0.5 GB)')
rir_count = len(os.listdir(RIR_DIR)) if os.path.exists(RIR_DIR) else 0
print(f'  OK MIT RIRs: {rir_count} files')

# FMA small dataset (~8 GB) — with retry + fallback
FMA_ZIP = '/content/fma_small.zip'
FMA_DIR = '/content/fma_small'  # zip extracts to fma_small/, not fma/
FMA_WAV = '/content/fma_wav'
FMA_OK = False
if not os.path.exists(FMA_DIR) or not os.path.exists(FMA_WAV) or len(os.listdir(FMA_WAV)) < 100:
    # Try downloading FMA small
    for attempt in range(3):
        if not os.path.exists(FMA_ZIP) or os.path.getsize(FMA_ZIP) < 7_000_000_000:
            print(f'  FMA download attempt {attempt+1}/3...')
            !wget --tries=3 --timeout=300 -O {FMA_ZIP} https://os.unil.cloud.switch.ch/fma/fma_small.zip
        if os.path.exists(FMA_ZIP) and os.path.getsize(FMA_ZIP) > 7_000_000_000:
            !unzip -q -o {FMA_ZIP} -d /content/
            # Delete the 8 GB zip after unzip to save disk space
            os.remove(FMA_ZIP)
            print('  Deleted FMA zip after unzip (saved ~8 GB)')
            FMA_OK = True
            break
        else:
            sz = os.path.getsize(FMA_ZIP) if os.path.exists(FMA_ZIP) else 0
            print(f'  FMA download incomplete: {sz/1e9:.2f} GB (need ~8 GB), retrying...')
            if os.path.exists(FMA_ZIP):
                os.remove(FMA_ZIP)
            time.sleep(10)
    if not FMA_OK:
        print('  WARNING: FMA download failed after 3 attempts.')
        print('  Will use ACAV100M + synthetic noise as background audio instead.')
else:
    FMA_OK = True
    # Clean up zip if it exists from a previous partial run
    if os.path.exists(FMA_ZIP):
        os.remove(FMA_ZIP)
# Verify the extracted directory actually exists
if FMA_OK and not os.path.exists(FMA_DIR):
    # Try alternative names
    for alt in ['/content/fma_small', '/content/fma', '/content/FMA']:
        if os.path.exists(alt):
            FMA_DIR = alt
            print(f'  FMA directory found at: {FMA_DIR}')
            break
    else:
        print(f'  WARNING: FMA unzipped but directory not found at {FMA_DIR}')
        # List what's in /content to debug
        dirs = [d for d in os.listdir('/content') if 'fma' in d.lower()]
        print(f'  Directories with "fma" in /content: {dirs}')
        FMA_OK = False
print(f'  OK FMA: {FMA_OK}')

# Convert FMA MP3s to WAVs (1500 clips) — only if FMA downloaded
if FMA_OK and (not os.path.exists(FMA_WAV) or len(os.listdir(FMA_WAV)) < 1000):
    os.makedirs(FMA_WAV, exist_ok=True)
    import glob, subprocess
    mp3s = sorted(glob.glob(f'{FMA_DIR}/**/*.mp3', recursive=True))[:1500]
    print(f'  Found {len(mp3s)} MP3 files in {FMA_DIR}, converting up to 1500...')
    converted = 0
    for mp3 in mp3s:
        wav = f'{FMA_WAV}/{os.path.splitext(os.path.basename(mp3))[0]}.wav'
        if not os.path.exists(wav) or os.path.getsize(wav) < 1000:
            subprocess.run(['ffmpeg', '-y', '-i', mp3, '-ar', '16000',
                           '-ac', '1', '-t', '30', wav],
                          capture_output=True, timeout=30)
            if os.path.exists(wav) and os.path.getsize(wav) > 1000:
                converted += 1
    print(f'  Converted {converted} MP3 → WAV')
fma_wav_count = len(os.listdir(FMA_WAV)) if os.path.exists(FMA_WAV) else 0
print(f'  OK FMA WAVs: {fma_wav_count} files')

# ACAV100M features (~17 GB) — with validation + fallback
# This is the largest download. On Colab free tier it may fail due to disk
# space or network timeouts. We handle this gracefully.
ACAV_SRC = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
ACAV_TRAIN = '/content/acav_train_subset.npy'
ACAV_VAL = '/content/acav_val_subset.npy'
ACAV_OK = False

# Check if we already have processed subsets (from a previous run)
if os.path.exists(ACAV_TRAIN) and os.path.exists(ACAV_VAL):
    ACAV_OK = True
    print(f'  OK ACAV subsets already processed (cached from previous run)')
elif os.path.exists(ACAV_SRC) and os.path.getsize(ACAV_SRC) > 1_000_000_000:
    # File exists and is > 1 GB — try to load it
    print(f'  ACAV source file exists: {os.path.getsize(ACAV_SRC)/1e9:.1f} GB')
    ACAV_OK = True
else:
    # Need to download
    # Clean up any partial/empty file first
    if os.path.exists(ACAV_SRC):
        sz = os.path.getsize(ACAV_SRC)
        print(f'  Removing incomplete ACAV file: {sz/1e6:.1f} MB (too small)')
        os.remove(ACAV_SRC)

    print('  Downloading ACAV100M features (~17 GB)... this takes 10-40 min')
    # Use huggingface_hub for better retry/resume support
    from huggingface_hub import hf_hub_download
    for attempt in range(3):
        try:
            result = hf_hub_download(
                repo_id='dscripka/openwakeword_features',
                filename='openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
                repo_type='dataset',
                local_dir='/content/',
            )
            # hf_hub_download saves to a cache; copy/symlink to expected path
            if os.path.exists(result) and os.path.getsize(result) > 1_000_000_000:
                if result != ACAV_SRC:
                    import shutil
                    shutil.move(result, ACAV_SRC)
                ACAV_OK = True
                print(f'  OK ACAV downloaded: {os.path.getsize(ACAV_SRC)/1e9:.1f} GB')
                break
            else:
                print(f'  ACAV download too small, retrying...')
        except Exception as e:
            print(f'  ACAV download attempt {attempt+1} failed: {e}')
            time.sleep(10)

    # Fallback: try wget if hf_hub_download failed
    if not ACAV_OK:
        print('  Trying wget fallback for ACAV...')
        !wget --tries=3 --timeout=600 -c -O {ACAV_SRC} \
            'https://huggingface.co/datasets/dscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
        if os.path.exists(ACAV_SRC) and os.path.getsize(ACAV_SRC) > 1_000_000_000:
            ACAV_OK = True

# Process ACAV into train/val subsets
if ACAV_OK and not (os.path.exists(ACAV_TRAIN) and os.path.exists(ACAV_VAL)):
    try:
        import numpy as np
        print('  Processing ACAV100M into train/val subsets...')
        acav = np.load(ACAV_SRC, mmap_mode='r')
        N = acav.shape[0]
        # Train: 80% of data, Val: 20%
        train_end = int(N * 0.8)
        np.save(ACAV_TRAIN, acav[:train_end].astype(np.float16))
        np.save(ACAV_VAL, acav[train_end:].astype(np.float16))
        del acav
        # Delete the 17 GB source file to save disk
        os.remove(ACAV_SRC)
        print('  Removed 17 GB ACAV source (subsets saved)')
    except Exception as e:
        print(f'  ERROR processing ACAV: {e}')
        ACAV_OK = False
        # Clean up corrupted file
        if os.path.exists(ACAV_SRC):
            os.remove(ACAV_SRC)

if ACAV_OK:
    train_sz = os.path.getsize(ACAV_TRAIN)/1e9 if os.path.exists(ACAV_TRAIN) else 0
    val_sz = os.path.getsize(ACAV_VAL)/1e6 if os.path.exists(ACAV_VAL) else 0
    print(f'  OK ACAV train: {train_sz:.1f} GB')
    print(f'  OK ACAV val: {val_sz:.0f} MB')
else:
    print('  WARNING: ACAV100M not available.')
    print('  Training will use FMA WAVs + synthetic noise as negatives instead.')
    print('  This produces slightly less robust models but still works.')

# Print disk usage summary
import subprocess
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print('\n--- Disk usage ---')
print(result.stdout.strip())

print('\n--- Shared data summary ---')
print(f'  OWW models:  melspectrogram + embedding (in resources dir ✓)')
print(f'  MIT RIRs:    {rir_count} files')
print(f'  FMA WAVs:    {fma_wav_count} files')
print(f'  ACAV100M:    {"available" if ACAV_OK else "NOT available (using fallback)"}')
print('All shared data ready.')



## 5. Command configuration — EDIT THIS TO CHANGE COMMANDS

Each command trains a separate OWW classifier. The shared infrastructure (Piper TTS, FMA, ACAV, RIRs) is reused across all commands.

Add or remove commands from this list. Each command trains in ~15-25 min.

In [ ]:
# ─── 4 CATEGORY MODELS TO TRAIN ─────────────────────────────────────
# Each entry trains a separate OWW classifier model that detects a
# command TYPE (not a specific app). The STT layer then extracts the
# parameter (which app, what search query).
#
# phrase:      a representative phrase for TTS clip generation
#              (the training loop generates clips for ALL apps × ALL verbs)
# model_name:  filename for the .onnx output (no spaces, lowercase)
# negatives:   phrases to exclude (other command types + soundalikes)
# intent:      the structured intent NEXUS will execute when detected
#
# ─── How it works ────────────────────────────────────────────────────
#
# 1. The acoustic model detects the command TYPE (open/close/search/play)
# 2. NEXUS records 1-2s of audio (shorter than full 3s)
# 3. STT transcribes the audio → "open gmail" → intent parser extracts "gmail"
# 4. NEXUS executes the command with the extracted parameter
#
# This gives ~300-800ms latency (vs ~500-1500ms for STT-only) and works
# for ANY app, even ones not in the training set.

import random
random.seed(42)  # reproducible sampling

COMMANDS = [
    # ═══════════════════════════════════════════════════════════════════
    # Model 1: OPEN — detects "open/launch/start <app>"
    # ═══════════════════════════════════════════════════════════════════
    {
        'phrase': 'open gmail',
        'model_name': 'command_open',
        # The training loop will generate ALL "open/launch/start <app>" phrases
        # as positive samples. Negatives are other command types + soundalikes.
        'negatives': [
            # Other command types (should NOT trigger "open" model)
            'close gmail', 'quit chrome', 'exit terminal', 'stop spotify',
            'search for gmail', 'find chrome', 'look up terminal',
            'play gmail', 'listen to spotify', 'play music',
            # Soundalike negatives
            'oh pen', 'oh pin', 'ope in', 'oppen', 'oh pan',
            # Non-command phrases
            'hey nexus', 'nexus', 'hello', 'yes', 'no',
        ],
        'intent': {'action': 'open_app', 'needs_param': True},
        # needs_param: True → after acoustic detection, record 1-2s + STT
        # to get the app name. The intent parser extracts it from the transcript.
    },

    # ═══════════════════════════════════════════════════════════════════
    # Model 2: CLOSE — detects "close/quit/exit <app>"
    # ═══════════════════════════════════════════════════════════════════
    {
        'phrase': 'close discord',
        'model_name': 'command_close',
        'negatives': [
            'open discord', 'launch chrome', 'start spotify',
            'search for discord', 'find chrome',
            'play discord', 'listen to music',
            'cloze', 'klose', 'clows', 'claws',
            'hey nexus', 'nexus', 'hello', 'yes', 'no',
        ],
        'intent': {'action': 'close_app', 'needs_param': True},
    },

    # ═══════════════════════════════════════════════════════════════════
    # Model 3: SEARCH — detects "search/find/look up <query>"
    # ═══════════════════════════════════════════════════════════════════
    {
        'phrase': 'search for cats',
        'model_name': 'command_search',
        'negatives': [
            'open cats', 'launch chrome', 'start spotify',
            'close cats', 'quit chrome', 'exit terminal',
            'play cats', 'listen to music',
            'serch', 'surch', 'circh', 'sirch',
            'hey nexus', 'nexus', 'hello', 'yes', 'no',
        ],
        'intent': {'action': 'search', 'needs_param': True},
    },

    # ═══════════════════════════════════════════════════════════════════
    # Model 4: PLAY — detects "play/listen to <media>"
    # ═══════════════════════════════════════════════════════════════════
    {
        'phrase': 'play music',
        'model_name': 'command_play',
        'negatives': [
            'open music', 'launch spotify', 'start youtube',
            'close music', 'quit spotify', 'exit youtube',
            'search for music', 'find spotify', 'look up youtube',
            'pley', 'pleigh', 'plai', 'plae', 'pray',
            'hey nexus', 'nexus', 'hello', 'yes', 'no',
        ],
        'intent': {'action': 'play', 'needs_param': True},
    },
]

# ─── Training Data Generation ───────────────────────────────────────
# For each category model, the training loop will generate TTS clips for
# a SUBSET of positive phrases (not all 960). The model learns the PATTERN
# "open <anything>" — it doesn't need every app name, just enough variety.
#
# Using 80 phrases per category gives good coverage of different verbs + apps
# while keeping clip generation to ~5 min per command (vs 1+ hour with 960).

MAX_PHRASES_PER_CATEGORY = 80  # sample subset for speed

# Map each command to a SAMPLED subset of category phrases
CATEGORY_MAP = {}
for cmd in COMMANDS:
    cat_key = cmd['model_name'].replace('command_', '')  # 'open', 'close', etc.
    all_phrases = CATEGORY_PHRASES[cat_key]
    if len(all_phrases) > MAX_PHRASES_PER_CATEGORY:
        sampled = random.sample(all_phrases, MAX_PHRASES_PER_CATEGORY)
    else:
        sampled = all_phrases
    CATEGORY_MAP[cmd['model_name']] = sampled

# ─── Summary ─────────────────────────────────────────────────────────
print(f'Commands to train: {len(COMMANDS)}')
print(f'  All are Type 2 (acoustic + STT parameter extraction)')
print(f'  Max phrases per category: {MAX_PHRASES_PER_CATEGORY} (sampled from full set)')
print()
for c in COMMANDS:
    cat_phrases = CATEGORY_MAP[c['model_name']]
    print(f'  {c["model_name"]:20s} ← "{c["phrase"]}"')
    print(f'    Training phrases: {len(cat_phrases)} (sampled)')
    print(f'    Negatives: {len(c["negatives"])}')
    print(f'    Intent: {c["intent"]}')
    print()

total_phrases = sum(len(p) for p in CATEGORY_MAP.values())
print(f'Total training phrases: {total_phrases}')
print(f'Estimated time: {len(COMMANDS) * 20} min ({len(COMMANDS) * 20 / 60:.1f} hrs)')
print(f'Output: {len(COMMANDS)} × ~800KB = ~{len(COMMANDS) * 0.8:.0f} MB total')
print(f'RAM impact: ~{len(COMMANDS) * 2} MB (models) + ~{len(COMMANDS) * 2} MB (buffers) = ~{len(COMMANDS) * 4} MB')

## 6. Training loop — trains each command sequentially

For each command:
1. Generate Piper TTS clips (positive = command phrase, negative = adversarial words)
2. Resample 22050 → 16000 Hz
3. Augment + extract features
4. Train DNN classifier (3-stage curriculum)
5. Ensemble + export ONNX
6. Download the `.onnx` file

The shared data (FMA, ACAV, RIRs) is loaded once and reused across all commands.

In [ ]:
import os, sys, copy, math, time, yaml, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.signal import resample_poly
from tqdm.auto import tqdm
import soundfile as sf
import glob

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ─── Check what data is available ─────────────────────────────────────
ACAV_TRAIN_PATH = '/content/acav_train_subset.npy'
ACAV_VAL_PATH = '/content/acav_val_subset.npy'
FMA_WAV_DIR = '/content/fma_wav'
RIR_DIR = '/content/mit_rirs'

ACAV_AVAILABLE = os.path.exists(ACAV_TRAIN_PATH) and os.path.exists(ACAV_VAL_PATH)
FMA_COUNT = len(os.listdir(FMA_WAV_DIR)) if os.path.exists(FMA_WAV_DIR) else 0
RIR_COUNT = len(os.listdir(RIR_DIR)) if os.path.exists(RIR_DIR) else 0

print(f'  ACAV100M: {"available" if ACAV_AVAILABLE else "NOT available — using synthetic negatives"}')
print(f'  FMA WAVs: {FMA_COUNT} files')
print(f'  MIT RIRs: {RIR_COUNT} files')


# ─── Load ACAV if available, else generate synthetic negatives ────────
if ACAV_AVAILABLE:
    acav_train_np = np.load(ACAV_TRAIN_PATH, mmap_mode='r')
    acav_val_np = np.load(ACAV_VAL_PATH)
    M = acav_val_np.shape[0]
    val_listen_hours = M * 0.08 / 3600.0
    n_win_val = M - 16
    acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
    acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
    print(f'  ACAV train (mmap): {acav_train_np.shape}')
    print(f'  ACAV val windows: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB, {val_listen_hours:.2f} hr)')
else:
    # Synthetic negatives: generate random feature-like data
    # This is a fallback — not as good as real ACAV100M, but allows training
    print('  Generating synthetic negative features (fallback)...')
    # Create ~100K random 16x96 windows as fake "ACAV" data
    SYNTH_N = 100_000
    acav_train_np = np.random.randn(SYNTH_N, 16, 96).astype(np.float32) * 0.5
    # Create a smaller val set (~10K windows)
    acav_val_np = np.random.randn(10_000, 16, 96).astype(np.float32) * 0.5
    M = acav_val_np.shape[0]
    val_listen_hours = M * 0.08 / 3600.0
    n_win_val = M - 16
    acav_val_windows = np.ascontiguousarray(acav_val_np.astype(np.float32))
    print(f'  Synthetic train: {acav_train_np.shape}')
    print(f'  Synthetic val: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB)')
    print('  WARNING: Models trained with synthetic negatives will have higher false positives.')
    print('           Retrain with real ACAV100M when possible for production use.')
VAL_BATCH = 4096

# ─── DNN model (same architecture as wake-word) ──────────────────────
class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

# ─── Helper: generate clips for one command ──────────────────────────
def generate_clips_for_command(cmd, n_samples=2000, n_samples_val=1000):
    """Generate Piper TTS clips for a command phrase.

    Builds a config YAML that openWakeWord's train.py requires.
    ALL required keys are always present — empty lists when data
    is unavailable (train.py handles empty lists gracefully).
    """
    model_name = cmd['model_name']
    phrase = cmd['phrase']
    negatives = cmd['negatives']
    output_dir = f'/content/{model_name}_output'

    # ─── Build config with ALL required keys ──────────────────────────
    # train.py accesses these keys unconditionally (no .get() with defaults).
    # If any key is missing, train.py crashes with KeyError.
    # We always include every key, using empty lists when data is unavailable.

    # Scale sample count based on number of target phrases
    # (more phrases = more diverse training data, but don't go overboard)
    target_phrases = CATEGORY_MAP.get(model_name, [phrase])
    n_phrases = len(target_phrases)
    # Fixed sample counts — no scaling needed since we already sampled phrases.
    # 2000 train + 500 val = 2500 clips = ~50 batches = ~5 min per command.
    # The model learns the PATTERN (e.g. "open <app>"), not specific apps,
    # so 2000 clips across 80 phrases (25 per phrase) is plenty of variety.
    n_samples_scaled = min(n_samples, 2000)
    n_samples_val_scaled = min(n_samples_val, 500)
    print(f'  Target phrases: {n_phrases}, samples: {n_samples_scaled}, val: {n_samples_val_scaled}')

    config = {
        # --- Clip generation ---
        # For category models, use ALL phrases from CATEGORY_MAP if available
        # (this generates TTS clips for all apps × all verbs in that category)
        'target_phrase': CATEGORY_MAP.get(model_name, [phrase]),
        'model_name': model_name,
        'custom_negative_phrases': negatives,
        'n_samples': n_samples_scaled,
        'n_samples_val': n_samples_val_scaled,
        'tts_batch_size': 50,
        'piper_sample_generator_path': '/content/piper-sample-generator',

        # --- Augmentation ---
        'augmentation_rounds': 1,
        'augmentation_batch_size': 16,
        # 'total_length' is computed by train.py from generated clips — not set here

        # --- Background noise (FMA) ---
        # train.py line 664: always accesses both keys.
        # Empty lists are fine — augment_clips() just skips background noise.
        'background_paths': [FMA_WAV_DIR] if FMA_COUNT > 100 else [],
        'background_paths_duplication_rate': [1] if FMA_COUNT > 100 else [],

        # --- Room impulse responses ---
        # train.py line 662: always accesses this key.
        # Empty list is fine — augment_clips() just skips RIR convolution.
        'rir_paths': [RIR_DIR] if RIR_COUNT > 0 else [],

        # --- Output ---
        'output_dir': output_dir,
        'onnx_export': True,
        'tflite_export': False,

        # --- Training (used by our custom training code, not train.py --train_model) ---
        'steps': 20000,
        'max_negative_weight': 1500,
        'target_accuracy': 0.7,
        'target_recall': 0.5,
        'target_false_positives_per_hour': 0.5,
        'batch_size': 128,
        'learning_rate': 1e-4,
        'model_type': 'dnn',
        'layer_dim': 128,
        'layer_size': 128,
        'n_blocks': 1,
        'model_input_shape': [16, 96],
        'n_classes': 1,
        'batch_n_per_class': {
            'adversarial_negative': 50,
            'positive': 50,
        },

        # --- ACAV features (for our custom training code) ---
        # train.py --train_model needs these, but we use our own training code.
        # We still include them so the config is complete if someone uses --train_model.
        'feature_data_files': {},
        'false_positive_validation_data_path': '',
    }

    # Add ACAV features if available
    if ACAV_AVAILABLE:
        config['batch_n_per_class']['ACAV100M_sample'] = 1024
        config['feature_data_files'] = {'ACAV100M_sample': ACAV_TRAIN_PATH}
        config['false_positive_validation_data_path'] = ACAV_VAL_PATH

    os.makedirs(output_dir, exist_ok=True)
    config_path = f'/content/{model_name}_config.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)
    return config, config_path

# ─── Helper: resample clips ──────────────────────────────────────────
def resample_clips(output_dir):
    TARGET_SR = 16000
    wav_dirs = sorted({os.path.dirname(f)
                       for f in glob.glob(f'{output_dir}/**/*.wav', recursive=True)})
    for d in wav_dirs:
        files = [f for f in os.listdir(d) if f.endswith('.wav')]
        if not files: continue
        sr = sf.info(f'{d}/{files[0]}').samplerate
        if sr == TARGET_SR: continue
        for f in files:
            p = f'{d}/{f}'
            data, sr = sf.read(p)
            if sr != TARGET_SR:
                new_data = resample_poly(data.astype('float32'), TARGET_SR, sr)
                sf.write(p, new_data, TARGET_SR)
    # Clear stale features
    for f in glob.glob(f'{output_dir}/**/*.npy', recursive=True):
        os.remove(f)

# ─── Helper: run train.py with error capture ─────────────────────────
def run_train_py(config_path, mode, timeout=600):
    """Run openWakeWord train.py with full error output.

    Returns (success, output_string).
    Captures ALL output (not just tail -5) so we can diagnose failures.
    """
    # CRITICAL: Set PYTHONPATH so the subprocess can find the openwakeword module.
    # The parent notebook has it in sys.path, but os.popen() subprocesses don't inherit that.
    env = os.environ.copy()
    oww_dir = '/content/openwakeword'
    if oww_dir not in env.get('PYTHONPATH', ''):
        env['PYTHONPATH'] = oww_dir + ':' + env.get('PYTHONPATH', '')
    cmd = f'PYTHONPATH={oww_dir} {sys.executable} /content/openwakeword/openwakeword/train.py --training_config {config_path} --{mode}'
    print(f'  running: {mode}...')
    try:
        result = os.popen(cmd + ' 2>&1').read()
        # Print last 10 lines for visibility, but keep full output
        lines = result.strip().split('\n')
        for line in lines[-10:]:
            print(f'    {line}')
        if 'Traceback' in result or 'Error' in result:
            return False, result
        return True, result
    except Exception as e:
        return False, str(e)

# ─── Helper: train one command ───────────────────────────────────────
def train_command(cmd, config, config_path):
    model_name = cmd['model_name']
    FEAT = config['feature_save_dir'] if 'feature_save_dir' in config else f"{config['output_dir']}/{model_name}"

    # Check if already trained (local)
    onnx_path = f'{FEAT}/{model_name}.onnx'
    if os.path.exists(onnx_path):
        print(f'  SKIP {model_name}: already trained locally ({onnx_path})')
        return onnx_path


    # ─── Step 1: Generate clips ───────────────────────────────────────
    dirs = {
        'positive_train': (f"{config['output_dir']}/{model_name}/positive_train", int(config['n_samples'] * 0.75)),
        'positive_test':  (f"{config['output_dir']}/{model_name}/positive_test", int(config['n_samples_val'] * 0.75)),
        'negative_train': (f"{config['output_dir']}/{model_name}/negative_train", int(config['n_samples'] * 0.75)),
        'negative_test':  (f"{config['output_dir']}/{model_name}/negative_test", int(config['n_samples_val'] * 0.75)),
    }
    all_full = all(os.path.isdir(p) and len(os.listdir(p)) >= exp for _, (p, exp) in dirs.items())
    if not all_full:
        print(f'  generating clips for "{cmd["phrase"]}"...')
        ok, output = run_train_py(config_path, 'generate_clips')
        if not ok:
            print(f'  ERROR: clip generation failed!')
            print(f'  Full output:\n{output}')
            raise RuntimeError(f'clip generation failed for {model_name}')

    # ─── Step 2: Resample 22050 → 16000 Hz ────────────────────────────
    print(f'  resampling clips to 16kHz...')
    resample_clips(config['output_dir'])

    # ─── Step 3: Augment + featurize ──────────────────────────────────
    needed = ['positive_features_train.npy', 'negative_features_train.npy',
              'positive_features_test.npy', 'negative_features_test.npy']
    if not all(os.path.exists(f'{FEAT}/{n}') for n in needed):
        print(f'  augmenting + featurizing...')
        ok, output = run_train_py(config_path, 'augment_clips')
        if not ok:
            print(f'  ERROR: augmentation failed!')
            print(f'  Full output:\n{output}')
            raise RuntimeError(f'augmentation failed for {model_name}')

    # Verify feature files exist
    for n in needed:
        path = f'{FEAT}/{n}'
        if not os.path.exists(path):
            raise RuntimeError(f'feature file missing: {path}')
        sz = os.path.getsize(path)
        if sz < 1000:
            raise RuntimeError(f'feature file too small ({sz} bytes): {path}')
        print(f'    {n}: {sz/1e6:.1f} MB')

    # ─── Step 4: Load features ────────────────────────────────────────
    pos_train = torch.from_numpy(np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)).to(DEVICE)
    neg_train = torch.from_numpy(np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)).to(DEVICE)
    pos_test  = torch.from_numpy(np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)).to(DEVICE)
    neg_test  = torch.from_numpy(np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)).to(DEVICE)
    print(f'  features: pos_train={tuple(pos_train.shape)} neg_train={tuple(neg_train.shape)}')
    # ─── Trim/pad features to exactly 16 time frames ──────────────────
    # The augmentation pipeline may produce more than 16 mel frames if the
    # TTS clips are slightly longer than expected. The model expects [N, 16, 96]
    # and the ACAV synthetic negatives are [N, 16, 96], so we must trim/pad
    # the positive and negative features to match.
    TARGET_FRAMES = 16

    def fix_time_dim(arr, target=TARGET_FRAMES):
        """Trim from center or zero-pad to exactly `target` time frames."""
        t = arr.shape[1]
        if t == target:
            return arr
        if t > target:
            # Trim from center — keeps the most informative middle frames
            start = (t - target) // 2
            return arr[:, start:start + target, :]
        # Pad with zeros at the end
        pad = torch.zeros(arr.shape[0], target - t, arr.shape[2],
                          device=arr.device, dtype=arr.dtype)
        return torch.cat([arr, pad], dim=1)

    pos_train = fix_time_dim(pos_train)
    neg_train = fix_time_dim(neg_train)
    pos_test  = fix_time_dim(pos_test)
    neg_test  = fix_time_dim(neg_test)
    print(f'  features (trimmed): pos_train={tuple(pos_train.shape)} neg_train={tuple(neg_train.shape)}')


    # ─── Step 5: Train ────────────────────────────────────────────────
    model = WakewordModel().to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    TOTAL_STEPS = config['steps']
    MAX_NEG_W = config['max_negative_weight']
    TARGET_FP = config['target_false_positives_per_hour']
    THRESH = 0.5
    history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
    best_models = []

    B_POS, B_ANEG, B_ACAV = 32, 32, 64
    def random_acav_window_batch(k):
        N, T, F = acav_train_np.shape
        rows = np.random.randint(0, N, size=k)
        if T == 16:
            # Synthetic data is already (N, 16, 96) — no windowing needed
            return acav_train_np[rows].astype(np.float32)
        starts = np.random.randint(0, T - 16 + 1, size=k)
        out = np.empty((k, 16, F), dtype=np.float32)
        for i, (r, s) in enumerate(zip(rows, starts)):
            out[i] = acav_train_np[r, s:s+16, :].astype(np.float32)
        return out

    def build_batch():
        p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
        aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
        p, an = pos_train[p_idx], neg_train[aneg_idx]
        acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
        x = torch.cat([p, an, acav], dim=0)
        y = torch.cat([torch.ones(B_POS, device=DEVICE),
                       torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
        return x, y

    @torch.no_grad()
    def validate(label):
        model.eval()
        p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
        n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
        recall = (p_preds >= THRESH).float().mean().item()
        accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                    / (pos_test.shape[0] + neg_test.shape[0]))
        n_fp = 0
        for i in range(0, n_win_val, VAL_BATCH):
            chunk = torch.from_numpy(acav_val_windows[i:i+VAL_BATCH]).to(DEVICE)
            n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
        fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
        history['val_recall'].append(recall)
        history['val_accuracy'].append(accuracy)
        history['val_fp_per_hour'].append(fp_per_hour)
        history['val_n_fp'].append(n_fp)
        save = False
        if len(history['val_n_fp']) >= 3:
            fp_p50 = np.percentile(history['val_n_fp'], 50)
            rc_p5 = np.percentile(history['val_recall'], 5)
            if n_fp <= fp_p50 and recall >= rc_p5:
                best_models.append((copy.deepcopy(model.state_dict()),
                                    {'val_recall': recall, 'val_accuracy': accuracy,
                                     'val_fp_per_hour': fp_per_hour, 'val_n_fp': n_fp}))
                save = True
        print(f'    [{label}] recall={recall:.3f} acc={accuracy:.3f} fp/hr={fp_per_hour:.2f} {"+" if save else "-"}', flush=True)
        model.train()

    def run_stage(idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
        optimizer = optim.Adam(model.parameters(), lr=lr)
        weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
        val_start = int(n_steps * (1.0 - val_window_frac))
        val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
        warmup = max(1, n_steps // 5)
        hold = n_steps // 3
        accumulated = []
        t0 = time.time()
        for step in range(n_steps):
            # Heartbeat to prevent Colab idle timeout
            heartbeat(f'training {model_name} stage {idx} step {step}/{n_steps}')
            if step < warmup: lr_now = lr * (step + 1) / warmup
            elif step < warmup + hold: lr_now = lr
            else:
                decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
                lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
            for pg in optimizer.param_groups: pg['lr'] = lr_now
            x, y = build_batch()
            logits = model(x).squeeze(-1)
            preds = torch.sigmoid(logits)
            keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
            if keep.sum() == 0:
                if step in val_steps: validate(f's{idx} {step}/{n_steps}')
                continue
            kept_logits = logits[keep]
            kept_y = y[keep]
            neg_w = weight_schedule[step]
            w = torch.where(kept_y > 0.5,
                            torch.tensor(1.0, device=DEVICE),
                            torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
            accumulated.append((kept_logits, kept_y, w))
            if sum(t[0].shape[0] for t in accumulated) >= 128:
                cat_logits = torch.cat([t[0] for t in accumulated])
                cat_y = torch.cat([t[1] for t in accumulated])
                cat_w = torch.cat([t[2] for t in accumulated])
                loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                history['loss'].append(loss.item())
                accumulated.clear()
            if step in val_steps:
                validate(f's{idx} {step}/{n_steps} ({(time.time()-t0)/60:.1f}m)')

    max_neg_w_now = MAX_NEG_W
    run_stage(1, TOTAL_STEPS, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP:
        max_neg_w_now *= 2
    run_stage(2, max(2000, TOTAL_STEPS // 10), lr=1e-5, max_neg_w=max_neg_w_now)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP:
        max_neg_w_now *= 2
    run_stage(3, max(2000, TOTAL_STEPS // 10), lr=1e-6, max_neg_w=max_neg_w_now)
    print(f'  training done: {len(best_models)} checkpoints, best fp/hr={min(history["val_fp_per_hour"]):.2f}')

    # ─── Step 6: Ensemble + export ────────────────────────────────────
    if not best_models:
        final_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        accs = [s['val_accuracy'] for _, s in best_models]
        rcs = [s['val_recall'] for _, s in best_models]
        fps = [s['val_fp_per_hour'] for _, s in best_models]
        acc_p90 = np.percentile(accs, 90)
        rc_p90 = np.percentile(rcs, 90)
        fp_p10 = np.percentile(fps, 10)
        qualified = [(sd, sc) for sd, sc in best_models
                     if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                     and sc['val_fp_per_hour'] <= fp_p10]
        if not qualified:
            qualified = [sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))[0]]
        keys = qualified[0][0].keys()
        final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    model.load_state_dict(final_state)
    model.eval()

    class WakewordExportable(nn.Module):
        def __init__(self, base): super().__init__(); self.base = base
        def forward(self, x): return torch.sigmoid(self.base(x))
    export_model = WakewordExportable(model).to(DEVICE).eval()
    dummy = torch.randn(1, 16, 96, device=DEVICE)
    torch.onnx.export(export_model, dummy, onnx_path,
                      input_names=['onnx::Flatten_0'], output_names=['output'],
                      dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                      opset_version=14, dynamo=False)
    print(f'  exported: {onnx_path} ({os.path.getsize(onnx_path)/1e3:.0f} KB)')

    # ─── Step 7: Sanity check ─────────────────────────────────────────
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)
    with torch.no_grad():
        p_scores = sess.run(None, {sess.get_inputs()[0].name: pos_test.cpu().numpy()})[0].flatten()
        print(f'  sanity: recall@0.5={(p_scores >= 0.5).mean():.3f}')


    # ─── Step 9: Clean up clips to save disk ──────────────────────────
    # Each command generates ~2GB of clips. With 39 commands that's 78GB.
    # Delete clips after features are extracted to avoid running out of disk.
    try:
        shutil.rmtree(f"{config['output_dir']}/{model_name}/positive_train", ignore_errors=True)
        shutil.rmtree(f"{config['output_dir']}/{model_name}/positive_test", ignore_errors=True)
        shutil.rmtree(f"{config['output_dir']}/{model_name}/negative_train", ignore_errors=True)
        shutil.rmtree(f"{config['output_dir']}/{model_name}/negative_test", ignore_errors=True)
        print(f'  cleaned up clips (saved ~2GB disk)')
    except Exception:
        pass  # Non-critical

    return onnx_path

# ─── PREFLIGHT: verify train.py can import everything ───────────────
# Run this BEFORE the training loop to catch import errors early
# (avoids wasting 20+ minutes on data download before discovering a broken import)
print("\n=== Preflight: verifying train.py imports ===")
preflight_code = (
    "from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator; "
    "from openwakeword.utils import compute_features_from_generator, AudioFeatures; "
    "print('train.py imports OK')"
)
preflight_cmd = f"PYTHONPATH=/content/openwakeword {sys.executable} -c \"{preflight_code}\""
preflight_result = os.popen(preflight_cmd + ' 2>&1').read()
print(preflight_result.strip())
if 'OK' not in preflight_result:
    print("PREFLIGHT FAILED: train.py cannot import required modules!")
    print("Fix the import errors above before running training.")
    raise RuntimeError("Preflight check failed — train.py imports broken")
print("=== Preflight passed ===\n")

# ─── MAIN LOOP: train all commands ───────────────────────────────────
trained_models = []
failed_commands = []
for i, cmd in enumerate(COMMANDS):
    print(f'\n{"="*70}')
    print(f'Command {i+1}/{len(COMMANDS)}: "{cmd["phrase"]}" → {cmd["model_name"]}')
    print(f'{"="*70}')
    heartbeat(f'starting command {i+1}/{len(COMMANDS)}: {cmd["model_name"]}')
    try:
        config, config_path = generate_clips_for_command(cmd)
        onnx_path = train_command(cmd, config, config_path)
        trained_models.append((cmd, onnx_path))

        # Also try browser download (works if tab is focused)
        try:
            from google.colab import files
            files.download(onnx_path)
            print(f'  browser download triggered: {os.path.basename(onnx_path)}')
        except Exception as e:
            print(f'  browser download skipped: {e}')
            print(f'  model saved at: {onnx_path} — download it before the session ends!')
    except Exception as e:
        print(f'  FAILED: {e}')
        failed_commands.append((cmd, str(e)))
        continue  # Don't abort the whole run — try the next command

print(f'\n{"="*70}')
print(f'DONE. Trained {len(trained_models)}/{len(COMMANDS)} command models.')
if failed_commands:
    print(f'FAILED: {len(failed_commands)} commands:')
    for cmd, err in failed_commands:
        print(f'  {cmd["model_name"]}: {err}')
print(f'{"="*70}')
for cmd, path in trained_models:
    print(f'  {cmd["model_name"]:20s} → {os.path.basename(path)} ({os.path.getsize(path)/1e3:.0f} KB)')
print(f'\nPlace all .onnx files at:')
print(f'  src-tauri/resources/oww/commands/')




## 7. Export intent mapping JSON

This creates a `command_intents.json` file that NEXUS loads at startup to map each command model to its intent.

In [ ]:
import json, os, shutil

# Build intent map from ALL commands (not just trained ones).
# This ensures the JSON is always complete, even if some commands failed.
# NEXUS will just skip models that aren't present on disk.
intent_map = {}
for cmd in COMMANDS:
    intent_map[cmd['model_name']] = {
        'phrase': cmd['phrase'],
        'model_file': f'{cmd["model_name"]}.onnx',
        'intent': cmd['intent'],
    }

json_path = '/content/command_intents.json'
with open(json_path, 'w') as f:
    json.dump(intent_map, f, indent=2)
print(f'Wrote {json_path} ({len(intent_map)} commands):')
print(json.dumps(intent_map, indent=2))



# Browser download
try:
    from google.colab import files
    files.download(json_path)
    print(f'\nBrowser download triggered: command_intents.json')
except Exception as e:
    print(f'Browser download skipped: {e}')
    print('Tip: re-run this cell to retry the download.')

print(f'\nPlace at: src-tauri/resources/oww/commands/command_intents.json')

## After Training

All 4 `.onnx` files and `command_intents.json` are saved in `/content/` and triggered as **browser downloads**. If a download doesn't fire automatically, re-run the training or export cell.

### Place files in the NEXUS project:

```
src-tauri/resources/oww/
├── nexus.onnx                    ← wake word (already works, no retraining needed)
├── melspectrogram.onnx           ← feature extractor (shared)
├── embedding_model.onnx          ← feature extractor (shared)
└── commands/
    ├── command_open.onnx         ← "open/launch/start <app>"
    ├── command_close.onnx        ← "close/quit/exit <app>"
    ├── command_search.onnx       ← "search/find/look up <query>"
    ├── command_play.onnx         ← "play/listen to <media>"
    └── command_intents.json      ← intent mapping
```

### Rebuild NEXUS:

```powershell
cd C:\PROJECTS\ULTRON
npm --prefix frontend run build
cd src-tauri
cargo build --release --features custom-protocol
```

### How it works at runtime:

1. User says **"NEXUS"** → wake word model fires (probability > 0.7)
2. NEXUS starts recording + runs 4 command models in parallel
3. User says **"open gmail"** → `command_open.onnx` fires
4. NEXUS records 1-2s more → STT transcribes → "open gmail"
5. Intent parser extracts: `action=open_app`, `target=gmail`
6. NEXUS executes → Gmail opens
7. If no command model fires → full 3s STT → intent parser → execute or send to backend

### Performance:

| Metric | Value |
|--------|-------|
| Wake → command detect | ~80ms (acoustic) |
| Command → execute | 300-800ms (STT parameter extraction) |
| Total latency | 380-880ms |
| RAM (4 models) | ~8 MB |
| CPU (4 inferences/80ms) | Negligible |
| Works for new apps | Yes (STT handles any app name) |